In [ ]:
# Source Code 5 (Google Colab environment)
# Script to train, test, and save a MobileNet model to classify aerial MHC images.

!pip install tensorflow keras opencv-python # Installs TensorFlow, Keras, and OpenCV libraries in the environment

import os  # Provides functions to interact with the operating system
import cv2  # Used for image processing tasks
import numpy as np  # Supports large, multi-dimensional arrays and matrices
import pandas as pd  # Used for data manipulation and analysis
from sklearn.model_selection import train_test_split  # Splits data into training and testing sets
from keras.models import Model  # Base class for defining Keras models
from keras.layers import Flatten, Dense, Dropout  # Layers for building neural networks
from keras.optimizers import Adam  # Adam optimizer for training models
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # Augments image data for training
from google.colab import drive  # Allows access to Google Drive from Colab


# Mount Google Drive
drive.mount('/content/drive')

# Set the paths to your image directories
mhc_dir = '/content/drive/MyDrive/mhc'
non_mhc_dir = '/content/drive/MyDrive/non_mhc'

# Load the images and labels
mhc_images = []
non_mhc_images = []
labels = []

# Load MHC images
for filename in os.listdir(mhc_dir):
    if filename.endswith(".png"):
        img = cv2.imread(os.path.join(mhc_dir, filename))
        img = cv2.resize(img, (224, 224)) # Resize image to 224x224
        mhc_images.append(img)
        labels.append(1) # Assign lab
    
# Load non-MHC images
for filename in os.listdir(non_mhc_dir):
    if filename.endswith(".png"):
        img = cv2.imread(os.path.join(non_mhc_dir, filename))
        img = cv2.resize(img, (224, 224)) # Resize image to 224x224
        non_mhc_images.append(img)
        labels.append(0) # Assign label 0 for Non-MHC images

# Convert lists to numpy arrays
mhc_images = np.array(mhc_images)
non_mhc_images = np.array(non_mhc_images)
labels = np.array(labels)

print("mhc_images shape:", mhc_images.shape)
print("non_mhc_images shape:", non_mhc_images.shape)
print("labels length:", len(labels))

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    np.concatenate((mhc_images, non_mhc_images), axis=0),
    labels,
    test_size=0.4,
    random_state=42
)

# Normalize pixel values to the range [0, 1]
X_train = X_train.astype('float32') / 255
X_test = X_test.astype('float32') / 255

# Load pre-trained MobileNet model (excluding the top layer)
base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the layers in the base model
for layer in base_model.layers:
 layer.trainable = False

# Add custom classification layers
x = base_model.output
x = Flatten()(x)
x = Dense(256, activation='relu')(x) # Adjust the number of neurons as needed
x = Dropout(0.5)(x) # Adding dropout for regularization
x = Dense(1, activation='sigmoid')(x)

# Create the model
model = Model(inputs=base_model.input, outputs=x)

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Train the model and store the history
model_history = model.fit(X_train, y_train, epochs=20, batch_size=64, validation_data=(X_test, y_test))

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test loss: {loss}")
print(f"Test accuracy: {accuracy}")

# Save the model for future use
model.save('/content/drive/MyDrive/mobilenet_mhc.h5')